## 1단계: 환경 준비 및 라이브러리 설치

- **수행 내용:** 파이썬에서 PostgreSQL 데이터베이스에 접속하기 위한 어댑터인 `psycopg2-binary` 설치.

In [1]:
%pip install psycopg2-binary

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------------------------- - 2.6/2.7 MB 21.6 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 19.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


## 2단계: 실시간 지오펜싱 시뮬레이션 (Real-time Mode)

- **수행 내용:** 실제 사람의 보행 속도(1.5m/s)를 반영하여 1초 단위로 좌표를 계산하고, 매초 DB 데이터와 하버사인 공식(거리 계산)을 비교하는 메인 스크립트 실행.

In [4]:

import time
import math
import psycopg2
from datetime import datetime

# ==============================================================================
# 1. Docker 기반 PostgreSQL 데이터베이스 연결 설정
# ==============================================================================
DB_CONFIG = {
    'dbname': 'postgres',
    'user': 'admin_user',
    'password': '1111',
    'host': 'localhost',
    'port': '5433'
}

# ==============================================================================
# 2. 하버사인 공식 (지구 곡률을 반영한 두 위경도 간의 정확한 거리(m) 계산 알고리즘)
# ==============================================================================
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = math.sin(delta_phi / 2.0)**2 + \
        math.cos(phi1) * math.cos(phi2) * \
        math.sin(delta_lambda / 2.0)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# ==============================================================================
# 3. 데이터베이스에서 locallink 스키마의 참조 데이터 실시간 Fetch
# ==============================================================================
def get_locations_from_db():
    try:
        print("🔗 Docker PostgreSQL(5433 포트)에 연결을 시도합니다...")
        conn = psycopg2.connect(**DB_CONFIG)
        cursor = conn.cursor()

        query = """
            SELECT name_ko, latitude, longitude, geofence_radius_m 
            FROM locallink.locations;
        """
        cursor.execute(query)

        # PostgreSQL NUMERIC/DECIMAL → float 변환으로 TypeError 방지
        locations = [
            (row[0], float(row[1]), float(row[2]), float(row[3]))
            for row in cursor.fetchall()
        ]

        cursor.close()
        conn.close()
        print(f"✅ DB 연결 성공! 총 {len(locations)}개의 지오펜싱 명소 데이터를 성공적으로 로드했습니다.\n")
        return locations

    except Exception as e:
        print(f"❌ 데이터베이스 연결 또는 쿼리 실행 중 에러가 발생했습니다: {e}")
        return []

# ==============================================================================
# 4. 외국인 관광객 가상 보행 동선 생성 및 실시간 지오펜싱 시뮬레이션
# ==============================================================================
def simulate_tourist_walk(sleep_interval=1.0):
    locations_ref = get_locations_from_db()
    if not locations_ref:
        print("데이터베이스에 명소 데이터가 존재하지 않아 시뮬레이션을 중단합니다.")
        return

    print("=========================================================================")
    print("🚶‍♂️ [시뮬레이션 시작] 외국인 관광객이 광화문 남단에서 북쪽을 향해 걷기 시작합니다.")
    print("=========================================================================\n")

    START_LAT, START_LON = 37.573000, 126.976800
    MID_LAT,   MID_LON   = 37.575900, 126.976900
    END_LAT,   END_LON   = 37.579617, 126.977041

    WALKING_SPEED_M_S = 1.5
    already_notified_locations = set()

    route_segments = [
        (START_LAT, START_LON, MID_LAT, MID_LON, "광화문 광장 방향"),
        (MID_LAT,   MID_LON,   END_LAT, END_LON,  "경복궁 근정전 방향")
    ]

    for start_lat, start_lon, end_lat, end_lon, direction in route_segments:
        print(f"\n📍 [경로 이동 중] {direction}으로 보행을 시작합니다...")

        segment_dist  = calculate_distance(start_lat, start_lon, end_lat, end_lon)
        total_seconds = int(segment_dist / WALKING_SPEED_M_S)

        # ★ 수정: range(total_seconds + 1) → 종점(ratio=1.0)까지 반드시 포함
        for step in range(total_seconds + 1):
            ratio       = min(step / total_seconds, 1.0) if total_seconds > 0 else 1.0
            current_lat = start_lat + (end_lat - start_lat) * ratio
            current_lon = start_lon + (end_lon - start_lon) * ratio

            current_time = datetime.now().strftime('%H:%M:%S')
            print(f"[{current_time}] 관광객 위치 ➔ 위도: {current_lat:.6f}, 경도: {current_lon:.6f}")

            for loc in locations_ref:
                loc_name, loc_lat, loc_lon, geofence_radius = loc
                distance_to_loc = calculate_distance(current_lat, current_lon, loc_lat, loc_lon)

                if distance_to_loc <= geofence_radius and loc_name not in already_notified_locations:
                    print("\n" + "🔥" * 35)
                    print(f"🚨 [지오펜스 진입 감지 이벤트 실시간 발생!] 🚨")
                    print(f" 🎯 대상 명소: {loc_name}")
                    print(f" 📏 진입 시점 거리: 약 {int(distance_to_loc)}m (설정 반경: {geofence_radius}m)")
                    print(f" ⚙️ 후속 액션 트리거: AI 도슨트 대본 생성 RAG 파이프라인 즉시 호출!")
                    print(f" 🤝 가이드 매칭: 반경 1km 이내 대기 중인 휴먼 가이드에게 매칭 핑 발송...")
                    print("🔥" * 35 + "\n")
                    already_notified_locations.add(loc_name)

            time.sleep(sleep_interval)

    print("\n🏁 시뮬레이션 동선이 모두 종료되었습니다. 시스템 테스트를 완료합니다.")

if __name__ == "__main__":
    simulate_tourist_walk()


🔗 Docker PostgreSQL(5433 포트)에 연결을 시도합니다...
✅ DB 연결 성공! 총 2개의 지오펜싱 명소 데이터를 성공적으로 로드했습니다.

🚶‍♂️ [시뮬레이션 시작] 외국인 관광객이 광화문 남단에서 북쪽을 향해 걷기 시작합니다.


📍 [경로 이동 중] 광화문 광장 방향으로 보행을 시작합니다...
[07:05:42] 관광객 위치 ➔ 위도: 37.573000, 경도: 126.976800
[07:05:43] 관광객 위치 ➔ 위도: 37.573013, 경도: 126.976800
[07:05:44] 관광객 위치 ➔ 위도: 37.573027, 경도: 126.976801
[07:05:45] 관광객 위치 ➔ 위도: 37.573040, 경도: 126.976801
[07:05:46] 관광객 위치 ➔ 위도: 37.573054, 경도: 126.976802
[07:05:47] 관광객 위치 ➔ 위도: 37.573067, 경도: 126.976802
[07:05:48] 관광객 위치 ➔ 위도: 37.573081, 경도: 126.976803
[07:05:49] 관광객 위치 ➔ 위도: 37.573094, 경도: 126.976803
[07:05:50] 관광객 위치 ➔ 위도: 37.573108, 경도: 126.976804
[07:05:51] 관광객 위치 ➔ 위도: 37.573121, 경도: 126.976804
[07:05:52] 관광객 위치 ➔ 위도: 37.573135, 경도: 126.976805
[07:05:53] 관광객 위치 ➔ 위도: 37.573148, 경도: 126.976805
[07:05:54] 관광객 위치 ➔ 위도: 37.573162, 경도: 126.976806
[07:05:55] 관광객 위치 ➔ 위도: 37.573175, 경도: 126.976806
[07:05:56] 관광객 위치 ➔ 위도: 37.573189, 경도: 126.976807
[07:05:57] 관광객 위치 ➔ 위도: 37.573202, 경도: 126.976807
[07:05:58] 관광객 위치 ➔ 위

## 3단계: 빠른 테스트 모드 (Fast Test Mode)

- **수행 내용:** 실제 시간으로 진행하면 약 10분 가까이 소요되는 시뮬레이션을 개발 및 디버깅 목적으로 빠르게 가속화한 커스텀 함수(`simulate_fast`) 실행.
- **설정 값:**
    - `time_step=10`: 10초 단위로 위치를 건너뛰며 계산
    - `sleep_interval=0.05`: 시스템 대기 시간을 0.05초로 단축하여 약 200배 가속

In [8]:

# ==============================================================================
# ⚡ 빠른 테스트 모드
#   - time_step      : 몇 초씩 건너뛰며 위치를 계산할지 (기본 10초 단위)
#   - sleep_interval : 각 스텝 사이 실제 대기 시간 (기본 0.05초 ≈ 200배 가속)
# ==============================================================================
def simulate_fast(time_step=10, sleep_interval=0.05):
    locations_ref = get_locations_from_db()
    if not locations_ref:
        print("❌ DB에 데이터가 없어 테스트를 중단합니다.")
        return

    print("=========================================================================")
    print(f"⚡ [빠른 테스트 모드]  스텝={time_step}초 / 대기={sleep_interval}초")
    print("=========================================================================\n")

    START_LAT, START_LON = 37.573000, 126.976800
    MID_LAT,   MID_LON   = 37.575900, 126.976900
    END_LAT,   END_LON   = 37.579617, 126.977041
    WALKING_SPEED_M_S    = 1.5
    already_notified     = set()

    route_segments = [
        (START_LAT, START_LON, MID_LAT, MID_LON, "광화문 광장 방향"),
        (MID_LAT,   MID_LON,   END_LAT, END_LON,  "경복궁 근정전 방향"),
    ]

    for start_lat, start_lon, end_lat, end_lon, direction in route_segments:
        print(f"\n📍 [{direction}] 이동 시작...")

        segment_dist  = calculate_distance(start_lat, start_lon, end_lat, end_lon)
        total_seconds = int(segment_dist / WALKING_SPEED_M_S)

        # time_step 단위로 건너뛰되, 종점(total_seconds)도 반드시 포함
        steps = list(range(0, total_seconds + 1, time_step))
        if steps[-1] != total_seconds:
            steps.append(total_seconds)

        for step in steps:
            ratio       = min(step / total_seconds, 1.0) if total_seconds > 0 else 1.0
            cur_lat     = start_lat + (end_lat - start_lat) * ratio
            cur_lon     = start_lon + (end_lon - start_lon) * ratio

            print(f"  [시뮬 {step:>4}초] 위도: {cur_lat:.6f}, 경도: {cur_lon:.6f}")

            for name, loc_lat, loc_lon, radius in locations_ref:
                d = calculate_distance(cur_lat, cur_lon, loc_lat, loc_lon)
                if d <= radius and name not in already_notified:
                    print("\n" + "🔥" * 30)
                    print(f"🚨 [지오펜스 진입!]  🎯 {name}")
                    print(f"   📏 거리: 약 {int(d)}m  (설정 반경: {radius}m)")
                    print(f"   ⚙️  RAG 파이프라인 트리거 / 🤝 가이드 매칭 핑 발송")
                    print("🔥" * 30 + "\n")
                    already_notified.add(name)

            time.sleep(sleep_interval)

    print("\n🏁 빠른 테스트 완료!")

# ▶ 실행 (time_step·sleep_interval 값을 바꿔 속도 조절 가능)
simulate_fast(time_step=10, sleep_interval=0.05)


🔗 Docker PostgreSQL(5433 포트)에 연결을 시도합니다...
✅ DB 연결 성공! 총 2개의 지오펜싱 명소 데이터를 성공적으로 로드했습니다.

⚡ [빠른 테스트 모드]  스텝=10초 / 대기=0.05초


📍 [광화문 광장 방향] 이동 시작...
  [시뮬    0초] 위도: 37.573000, 경도: 126.976800
  [시뮬   10초] 위도: 37.573135, 경도: 126.976805
  [시뮬   20초] 위도: 37.573270, 경도: 126.976809
  [시뮬   30초] 위도: 37.573405, 경도: 126.976814
  [시뮬   40초] 위도: 37.573540, 경도: 126.976819
  [시뮬   50초] 위도: 37.573674, 경도: 126.976823
  [시뮬   60초] 위도: 37.573809, 경도: 126.976828
  [시뮬   70초] 위도: 37.573944, 경도: 126.976833
  [시뮬   80초] 위도: 37.574079, 경도: 126.976837
  [시뮬   90초] 위도: 37.574214, 경도: 126.976842
  [시뮬  100초] 위도: 37.574349, 경도: 126.976847
  [시뮬  110초] 위도: 37.574484, 경도: 126.976851
  [시뮬  120초] 위도: 37.574619, 경도: 126.976856
  [시뮬  130초] 위도: 37.574753, 경도: 126.976860
  [시뮬  140초] 위도: 37.574888, 경도: 126.976865
  [시뮬  150초] 위도: 37.575023, 경도: 126.976870

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🚨 [지오펜스 진입!]  🎯 광화문 광장
   📏 거리: 약 97m  (설정 반경: 100.0m)
   ⚙️  RAG 파이프라인 트리거 / 🤝 가이드 매칭 핑 발송
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥

  [시뮬  16